
# Assignment 4 - NER NEL

Use GliNER and Wikipedia search to perform entity linking with specific labels on domain-specific documents.

## Setup and Useful functions

In [4]:
!pip install requests gliner gatenlp[formats]


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import requests
from gliner import GLiNER
from pprint import pprint
from gatenlp import Document

In [6]:
# -----------------------------
# Wikipedia utilities
# -----------------------------

def to_wiki_url(title):
    if title is None:
        return None
    return "https://en.wikipedia.org/wiki/" + title.replace(" ", "_")

labels = ["person", "organization", "location", "team", "event"]

def wikipedia_link(entity):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": entity,
        "format": "json",
        "utf8": 1
    }

    headers = {
        "User-Agent": "nel-demo/1.0"
    }

    r = requests.get(url, params=params, headers=headers, timeout=5)

    if r.status_code != 200:
        return None

    try:
        data = r.json()
    except Exception:
        return None

    results = data.get("query", {}).get("search", [])
    if not results:
        return None

    return results[0]["title"]


## CyberSecurity Example

In [7]:
text_cyber = """The software ecosystem is continuously exposed to critical vulnerabilities
affecting widely deployed systems. Notable historical examples include
Heartbleed, Rowhammer, and Spectre, all of which demonstrated severe
hardware and software security flaws impacting modern computing
infrastructures. A recent major vulnerability, Log4Shell (CVE-2021-44228),
in the Apache Log4j library, enabled remote code execution through malicious
log message injection in Java applications.\n
\n
Following disclosure events, a rapid response cycle emerges involving
attackers, system operators, security vendors, and researchers. Attackers
attempt exploitation before patch deployment, while organizations race to
update vulnerable systems. Monitoring and mitigation efforts often involve
threat intelligence and vulnerability databases such as the National
Vulnerability Database (NVD).\n
\n
Log4Shell exploits the Java Naming and Directory Interface (JNDI),
leveraging services such as LDAP and RMI to trigger remote lookups that can
lead to code execution. The vulnerability spread across dependencies in
ecosystems such as Maven Central, affecting thousands of software packages
indirectly. Security research institutions and companies such as Google
analyzed its global impact and propagation patterns.\n
\n
To understand exploitation dynamics, researchers deployed network telescopes
to collect traffic related to scanning activity targeting HTTP-based
services. This allowed analysis of payload distribution, exploit attempts,
and attacker behavior following the initial disclosure of Log4Shell.\n
"""

gold_cyber = [
    "https://en.wikipedia.org/wiki/Heartbleed",
    "https://en.wikipedia.org/wiki/Rowhammer",
    "https://en.wikipedia.org/wiki/Spectre_(security_vulnerability)",
    "https://en.wikipedia.org/wiki/Log4Shell",
    "https://en.wikipedia.org/wiki/CVE-2021-44228",
    "https://en.wikipedia.org/wiki/Apache_Log4j",
    "https://en.wikipedia.org/wiki/Maven_(software)",
    "https://en.wikipedia.org/wiki/Java_Naming_and_Directory_Interface",
    "https://en.wikipedia.org/wiki/LDAP",
    "https://en.wikipedia.org/wiki/Arbitrary_code_execution",
]

label_cyber = [
    "software",
    "protocol",
    "vulnerability",
    "attack",
]

## Biomedical Example

In [8]:
text_bio = """Modern biomedical research increasingly relies on computational methods to
understand disease mechanisms and molecular interactions. A key focus is the
study of SARS-CoV-2, the virus responsible for COVID-19, and its interaction
with the human ACE2 receptor, which plays a critical role in viral entry
into host cells.\n
\n
Experimental studies have shown that genes such as ACE2 and TMPRSS2 are
highly relevant for infection susceptibility. These proteins facilitate
viral binding and membrane fusion, enabling infection of epithelial cells in
the respiratory tract.\n
\n
To analyze these mechanisms, researchers often use single-cell RNA
sequencing (scRNA-seq) to profile gene expression at cellular resolution.
Techniques such as CRISPR-Cas9 gene editing are employed to knock out
specific genes like ACE2 in order to observe functional changes in viral
infectivity.\n
\n
Computational models, including transformer-based architectures, are
increasingly used to predict disease progression in COVID-19 patients. These
models integrate clinical and molecular data to improve diagnosis and
treatment strategies.\n
\n
Interferon signaling pathways have also been identified as key components of
the immune response to viral infection, influencing disease severity and
outcomes.\n
"""

gold_bio = [
    "https://en.wikipedia.org/wiki/COVID-19",
    "https://en.wikipedia.org/wiki/SARS-CoV-2",
    "https://en.wikipedia.org/wiki/Angiotensin-converting_enzyme_2",
    "https://en.wikipedia.org/wiki/TMPRSS2",
    "https://en.wikipedia.org/wiki/SnRNA-seq",
    "https://en.wikipedia.org/wiki/Cas9",
    "https://en.wikipedia.org/wiki/Interferon"
]

label_bio = [
    "disease",
    "protein",
    "method",
]

# Exercise 1 :

## Apply GliNER to the two documents considering the respective labels.

In [9]:

model = GLiNER.from_pretrained(
    "urchade/gliner_medium-v2.1"
)


C:\Users\broba\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
2026-06-12 00:50:19,000|INFO|httpx|HTTP Request: GET https://huggingface.co/api/models/urchade/gliner_medium-v2.1/revision/main "HTTP/1.1 200 OK"
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 9141.90it/s]
2026-06-12 00:50:19,037|INFO|gliner.model|Loading the following GLiNER type: <class 'gliner.model.UniEncoderSpanGLiNER'>...
2026-06-12 00:50:19,218|INFO|httpx|HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-12 00:50:19,285|INFO|httpx|HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-base/8ccc9b6f36199bec6961081d44eb72fb3f7353f3/config.json "HTTP/1.1 200 OK"
2026-06-12 00:50:19,478|INFO|httpx

In [10]:
labels_cyber = [
    "software",
    "vulnerability",
    "protocol",
    "organization",
    "database",
    "programming language"
]

In [11]:
labels_bio = [
    "virus",
    "disease",
    "gene",
    "protein",
    "biological process",
    "technology"
]

In [12]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

In [13]:
from gliner import GLiNER

model = GLiNER.from_pretrained(
    "urchade/gliner_medium-v2.1"
)

2026-06-12 00:50:29,879|INFO|httpx|HTTP Request: GET https://huggingface.co/api/models/urchade/gliner_medium-v2.1/revision/main "HTTP/1.1 200 OK"
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 11625.01it/s]
2026-06-12 00:50:29,916|INFO|gliner.model|Loading the following GLiNER type: <class 'gliner.model.UniEncoderSpanGLiNER'>...
2026-06-12 00:50:30,096|INFO|httpx|HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-12 00:50:30,163|INFO|httpx|HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-base/8ccc9b6f36199bec6961081d44eb72fb3f7353f3/config.json "HTTP/1.1 200 OK"
2026-06-12 00:50:30,403|INFO|httpx|HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v3-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-12 00:50:30,466|INFO|httpx|HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v3-base/8c

In [14]:
entities_cyber = model.predict_entities(
    text_cyber,
    labels_cyber
)

In [15]:
print("CYBER ENTITIES\n")

for ent in entities_cyber:
    pprint(ent)

CYBER ENTITIES

{'end': 156,
 'label': 'vulnerability',
 'score': 0.8792765736579895,
 'start': 146,
 'text': 'Heartbleed'}
{'end': 167,
 'label': 'vulnerability',
 'score': 0.8115765452384949,
 'start': 158,
 'text': 'Rowhammer'}
{'end': 180,
 'label': 'vulnerability',
 'score': 0.7750232219696045,
 'start': 173,
 'text': 'Spectre'}
{'end': 335,
 'label': 'vulnerability',
 'score': 0.8606425523757935,
 'start': 326,
 'text': 'Log4Shell'}
{'end': 351,
 'label': 'vulnerability',
 'score': 0.8472442626953125,
 'start': 337,
 'text': 'CVE-2021-44228'}
{'end': 381,
 'label': 'software',
 'score': 0.7760135531425476,
 'start': 361,
 'text': 'Apache Log4j library'}
{'end': 460,
 'label': 'programming language',
 'score': 0.9924284815788269,
 'start': 456,
 'text': 'Java'}
{'end': 862,
 'label': 'database',
 'score': 0.8577159643173218,
 'start': 831,
 'text': 'National\nVulnerability Database'}
{'end': 882,
 'label': 'vulnerability',
 'score': 0.7196945548057556,
 'start': 873,
 'text': 'Log

In [16]:
entities_bio = model.predict_entities(
    text_bio,
    labels_bio
)

In [17]:
print("BIO ENTITIES\n")

for ent in entities_bio:
    pprint(ent)

BIO ENTITIES

{'end': 171,
 'label': 'virus',
 'score': 0.9765180945396423,
 'start': 161,
 'text': 'SARS-CoV-2'}
{'end': 207,
 'label': 'disease',
 'score': 0.9700560569763184,
 'start': 199,
 'text': 'COVID-19'}
{'end': 248,
 'label': 'gene',
 'score': 0.5405192971229553,
 'start': 244,
 'text': 'ACE2'}
{'end': 377,
 'label': 'gene',
 'score': 0.7957268357276917,
 'start': 373,
 'text': 'ACE2'}
{'end': 389,
 'label': 'gene',
 'score': 0.7961691617965698,
 'start': 382,
 'text': 'TMPRSS2'}
{'end': 438,
 'label': 'biological process',
 'score': 0.5011986494064331,
 'start': 414,
 'text': 'infection susceptibility'}
{'end': 479,
 'label': 'biological process',
 'score': 0.6968205571174622,
 'start': 466,
 'text': 'viral binding'}
{'end': 499,
 'label': 'biological process',
 'score': 0.8035608530044556,
 'start': 484,
 'text': 'membrane fusion'}
{'end': 646,
 'label': 'technology',
 'score': 0.6732951998710632,
 'start': 620,
 'text': 'single-cell RNA\nsequencing'}
{'end': 740,
 'label'

## Visualize the annotated documents (e.g., with GateNLP).

In [18]:
from IPython.display import HTML, display

def visualize_entities(text, entities):

    entities = sorted(
        entities,
        key=lambda x: x["start"],
        reverse=True
    )

    html = text

    for ent in entities:

        start = ent["start"]
        end = ent["end"]

        label = ent["label"]

        entity_text = html[start:end]

        replacement = (
            f"<mark>{entity_text}"
            f" <b>[{label}]</b></mark>"
        )

        html = (
            html[:start]
            + replacement
            + html[end:]
        )

    display(
        HTML(html.replace("\n", "<br>"))
    )

In [19]:
visualize_entities(
    text_cyber,
    entities_cyber
)

In [20]:
visualize_entities(
    text_bio,
    entities_bio
)

# Exercise 2 :

## Perform NEL with Wikipedia search engine (you can use the provided function wikipedia link) on the entity mentions given by GliNER.

In [21]:
print("CYBER NEL RESULTS\n")

for ent in entities_cyber:

    mention = ent["text"]

    wiki_title = wikipedia_link(
        mention
    )

    wiki_url = to_wiki_url(
        wiki_title
    )

    print(
        mention,
        "->",
        wiki_url
    )

CYBER NEL RESULTS

Heartbleed -> https://en.wikipedia.org/wiki/Heartbleed
Rowhammer -> https://en.wikipedia.org/wiki/Row_hammer
Spectre -> https://en.wikipedia.org/wiki/SPECTRE
Log4Shell -> https://en.wikipedia.org/wiki/Log4Shell
CVE-2021-44228 -> https://en.wikipedia.org/wiki/Log4Shell
Apache Log4j library -> https://en.wikipedia.org/wiki/Log4j
Java -> https://en.wikipedia.org/wiki/Java
National
Vulnerability Database -> https://en.wikipedia.org/wiki/National_Vulnerability_Database
Log4Shell -> https://en.wikipedia.org/wiki/Log4Shell
Java -> https://en.wikipedia.org/wiki/Java
JNDI -> None
LDAP -> None
RMI -> None
Security research institutions -> None
Google -> None
Log4Shell -> None


In [22]:
print("BIO NEL RESULTS\n")

for ent in entities_bio:

    mention = ent["text"]

    wiki_title = wikipedia_link(
        mention
    )

    wiki_url = to_wiki_url(
        wiki_title
    )

    print(
        mention,
        "->",
        wiki_url
    )

BIO NEL RESULTS

SARS-CoV-2 -> https://en.wikipedia.org/wiki/SARS-CoV-2
COVID-19 -> https://en.wikipedia.org/wiki/COVID-19
ACE2 -> https://en.wikipedia.org/wiki/Angiotensin-converting_enzyme_2
ACE2 -> https://en.wikipedia.org/wiki/Angiotensin-converting_enzyme_2
TMPRSS2 -> https://en.wikipedia.org/wiki/TMPRSS2
infection susceptibility -> https://en.wikipedia.org/wiki/Norovirus
viral binding -> https://en.wikipedia.org/wiki/Viral_replication
membrane fusion -> https://en.wikipedia.org/wiki/Membrane_fusion_protein
single-cell RNA
sequencing -> https://en.wikipedia.org/wiki/Single-cell_sequencing
CRISPR-Cas9 -> https://en.wikipedia.org/wiki/CRISPR_gene_editing
ACE2 -> None
viral
infectivity -> None
Computational models -> None
transformer-based architectures -> None
disease progression -> None
COVID-19 -> https://en.wikipedia.org/wiki/COVID-19
Interferon signaling pathways -> https://en.wikipedia.org/wiki/JAK-STAT_signaling_pathway
immune response -> https://en.wikipedia.org/wiki/Immune_r

# Exercise 3 :  Evaluate the results of NEL

In [24]:
gold_cyber = {
    "Heartbleed",
    "Rowhammer",
    "Spectre",
    "Log4Shell",
    "Apache Log4j",
    "Java",
    "LDAP",
    "RMI",
    "JNDI",
    "Google",
    "National Vulnerability Database"
}

In [25]:
gold_bio = {
    "SARS-CoV-2",
    "COVID-19",
    "ACE2",
    "TMPRSS2",
    "infection susceptibility",
    "viral binding",
    "membrane fusion",
    "single-cell RNA sequencing",
    "CRISPR-Cas9",
    "disease progression"
}

In [26]:
pred_cyber = {
    ent["text"]
    for ent in entities_cyber
}

In [27]:
pred_bio = {
    ent["text"]
    for ent in entities_bio
}

In [28]:
def evaluate(predicted, gold):

    tp = len(
        predicted.intersection(gold)
    )

    precision = tp / len(predicted)

    recall = tp / len(gold)

    f1 = (
        2 * precision * recall
        /
        (precision + recall)
    )

    return precision, recall, f1

In [29]:
p, r, f1 = evaluate(
    pred_cyber,
    gold_cyber
)

print("CYBER")
print("Precision =", p)
print("Recall =", r)
print("F1 =", f1)

CYBER
Precision = 0.6923076923076923
Recall = 0.8181818181818182
F1 = 0.7500000000000001


In [30]:
p, r, f1 = evaluate(
    pred_bio,
    gold_bio
)

print("BIO")
print("Precision =", p)
print("Recall =", r)
print("F1 =", f1)

BIO
Precision = 0.6
Recall = 0.9
F1 = 0.7200000000000001


In [31]:
print(pred_cyber)

{'Java', 'Security research institutions', 'Log4Shell', 'Spectre', 'National\nVulnerability Database', 'JNDI', 'Google', 'Rowhammer', 'Heartbleed', 'Apache Log4j library', 'LDAP', 'RMI', 'CVE-2021-44228'}


In [32]:
print(pred_bio)

{'Interferon signaling pathways', 'SARS-CoV-2', 'single-cell RNA\nsequencing', 'disease progression', 'transformer-based architectures', 'Computational models', 'CRISPR-Cas9', 'infection susceptibility', 'TMPRSS2', 'immune response', 'COVID-19', 'membrane fusion', 'viral binding', 'viral\ninfectivity', 'ACE2'}
